In [6]:
import metapredict as mp
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import statistics
from Bio.SeqUtils import ProtParam

In [7]:
tf = pd.read_csv('BP2.csv').dropna()
tf.reset_index(inplace=True)
#tf[tf.isna().any(axis=1)]
#tf=tf.dropna()

In [8]:
len(tf)

1246

In [9]:

#Individual hydropathy score dictionary
hp ={'A':1.8, 'R':-4.5, 'N':-3.5, 'D':-3.5, 'C': 2.5, 'Q':-3.5, 'E':-3.5,
    'G':-0.4, 'H':-3.2, 'I': 4.5, 'L': 3.8, 'K':-3.9, 'M': 1.9, 'F': 2.8,
    'P':-1.6, 'S':-0.8, 'T':-0.7, 'W':-0.9, 'Y':-1.3, 'V': 4.2}

n_hp = {key: (value - min(hp.values())) / (max(hp.values()) - min(hp.values())) for key, value in hp.items()}

def fractions(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    aa_counts = prot_param.count_amino_acids()
    total_aa_count = sum(aa_counts.values())
    aa_fractions = {aa: count / total_aa_count for aa, count in aa_counts.items()}

    return aa_fractions

def n_hp_score(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    aa_counts = prot_param.count_amino_acids()
    total_aa_count = sum(aa_counts.values())
    aa_fractions = {aa: count / total_aa_count for aa, count in aa_counts.items()}
    hpscore = {amino_acid: n_hp[amino_acid] * aa_fractions[amino_acid] for amino_acid in n_hp}
    hpscore = sum(hpscore.values())
    return hpscore

def pondr_score(sequence):
    # Create a SeqRecord from the sequence in the DataFrame
    seq_record = SeqRecord(Seq(sequence), id="1")
    
    # Calculate the PONDR score for each sequence
    score = mp.predict_disorder(seq_record.seq)
    average = statistics.mean(score)
    return average

def calculate_net_charge(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    net_charge = prot_param.charge_at_pH(7.4)  # Assuming physiological pH is 7.4
    return net_charge

def hp_score(sequence):
    prot_param = ProtParam.ProteinAnalysis(sequence)
    hp = prot_param.gravy()  # Assuming physiological pH is 7.4
    return hp

#tf['NetCharge'] = tf['BP Sequence'].apply(calculate_net_charge)

#tf['HP Score'] = tf['BP Sequence'].apply(hp_score)
#print(tf['BP Sequence'])

tf['Fractions'] = tf['BP Sequence'].apply(fractions)
columns_order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
df_aa_fractions = pd.DataFrame(tf['Fractions'].apply(lambda x: [x[aa] for aa in columns_order]).tolist(), columns=columns_order)

tf['Normalized HP Score'] = tf['BP Sequence'].apply(n_hp_score)
tf['PONDR Score'] = tf['BP Sequence'].apply(pondr_score)

new = pd.DataFrame({
    'Protein': tf['Protein'],
    'UniProtID': tf['UniProt ID'],
    'Label': tf['Label'],
    'BP Sequence': tf['BP Sequence'],
    'Normalized HP Score' : tf['Normalized HP Score'],
    'PONDR Score' : tf['PONDR Score']
})

result = pd.concat([new, df_aa_fractions], axis=1)

result['f+']= result['R'] + result['K']
result['f-']= result['D'] + result['E']
result['|f+ - f-|']= abs(result['f+'] - result['f-'])

result.to_csv('BP2_properties.csv',index=False)
print(len(result))

1246


In [10]:
result

,Protein,UniProtID,Label,BP Sequence,Normalized HP Score,PONDR Score,A,C,D,E,...,Q,R,S,T,V,W,Y,f+,f-,|f+ - f-|
0,ANKZ1_HUMAN,Q9H8Y5,1.0,MSPAPDAAPAPASISLFDLSADAPVFQGLSLVSHAPGEALARAPRT...,0.472300,0.826318,0.154930,0.014085,0.056338,0.070423,...,0.028169,0.056338,0.154930,0.014085,0.028169,0.000000,0.000000,0.084507,0.126761,0.042254
1,ANM3_HUMAN,O60678,1.0,MCSLASGATGGRGAVENEEDLPELSDSGDEAAWEDEDDADLPHGKQQ,0.389125,0.856934,0.127660,0.021277,0.148936,0.148936,...,0.042553,0.021277,0.085106,0.021277,0.021277,0.021277,0.000000,0.042553,0.297872,0.255319
2,CHAP1_HUMAN,Q96JM3,1.0,MEAFQELRKPSARLECDHCSFRGTDYENVQIHMGTIHPEFCDEMDA...,0.404674,0.765160,0.065129,0.016282,0.037992,0.082768,...,0.033921,0.031208,0.147897,0.032564,0.042062,0.020353,0.012212,0.131615,0.120760,0.010855
3,DZIP1_HUMAN,Q86YF9,1.0,MQAEAADWFSSMPFQKHVYYPLASGPEGPDVAVAAAAAGAASMACA...,0.477214,0.289516,0.116751,0.030457,0.045685,0.076142,...,0.060914,0.045685,0.071066,0.040609,0.050761,0.010152,0.020305,0.116751,0.121827,0.005076
4,LMBL1_HUMAN,Q9Y468,1.0,MHLVAGDSPGSGPHLPATAFIIPASSATLGLPSSALDVSCFPREPI...,0.423620,0.505674,0.075163,0.024510,0.058824,0.083333,...,0.049020,0.037582,0.098039,0.040850,0.053922,0.024510,0.026144,0.080065,0.142157,0.062092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1241,GTD2A_HUMAN,Q86UP8,45.0,MAQVAVSTLPVEEESSSETRMVVTFLVSALESMCKELAKSKAEVAC...,0.525888,0.217399,0.113402,0.061856,0.030928,0.113402,...,0.020619,0.030928,0.072165,0.061856,0.154639,0.000000,0.020619,0.092784,0.144330,0.051546
1242,GTD2B_HUMAN,Q6EKJ0,45.0,MAQVAVSTLPVEEESSSETRMVVTFLVSALESMCKELAKSKAEVAC...,0.525888,0.217399,0.113402,0.061856,0.030928,0.113402,...,0.020619,0.030928,0.072165,0.061856,0.154639,0.000000,0.020619,0.092784,0.144330,0.051546
1243,MBNL2_HUMAN,Q5VZF2,1.0,HLKTQLEINGRNNLIQQKTAAAMLAQQMQFMFPGTPLHPVPTFPVG...,0.523638,0.776070,0.088235,0.000000,0.000000,0.019608,...,0.068627,0.019608,0.029412,0.127451,0.088235,0.000000,0.009804,0.049020,0.019608,0.029412
1244,COE2_HUMAN,Q9HAK2,42.0,MFGIQDTLGRGPTLKEKSLGAEMDSVRSWVRNVGVVDANVAAQSGV...,0.436861,0.248535,0.051587,0.027778,0.055556,0.059524,...,0.047619,0.079365,0.063492,0.051587,0.095238,0.003968,0.019841,0.138889,0.115079,0.023810


In [ ]:
counts_column1 = tf['Annotation Score'].value_counts()
counts_column2 = tf['Existence'].value_counts()

# Create histograms
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.bar(counts_column1.index, counts_column1.values)
plt.grid()
plt.title('Annotation Score')

plt.subplot(1, 2, 2)
plt.bar(counts_column2.index, counts_column2.values)
plt.title('Existence')
plt.xticks(rotation=90)
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(tf['NetCharge'], bins=50, edgecolor='black')
plt.title('Net Charge Distribution')

# Plot histogram for sequence length
tf['SequenceLength'] = tf['Sequence'].apply(len)
plt.subplot(1, 2, 2)
plt.hist(tf['SequenceLength'], bins=50, edgecolor='black')
plt.title('Sequence Length Distribution')

plt.tight_layout()
plt.show()

In [ ]:
unique_domains = tf['Domain'].unique().tolist()
unique_types = df['Type'].unique().tolist()

# Create a new DataFrame from the unique values
unique_df = pd.DataFrame({'Unique Domains': unique_domains, 'Unique Types': unique_types})

# Write the DataFrame to a CSV file
unique_df.to_csv('unique_values.csv', index=False)